# Notebook 03.1: RAG + Query Rewriting (OpenAI GPT-4.1-mini)

**Pipeline:** Query -> LLM Rewrite -> BM25 Retrieval -> LLM Generate -> yes/no/maybe
**Evaluasi:** 4 metrik RAGAS custom zero-NaN

## 1. Impor Library

In [1]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime
from collections import Counter

from openai import OpenAI
from rank_bm25 import BM25Okapi
from datasets import load_dataset

warnings.filterwarnings('ignore')
print('Semua library berhasil diimpor!')
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | Pandas: {pd.__version__}')

C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Semua library berhasil diimpor!
Python: 3.11.9 | NumPy: 2.3.5 | Pandas: 2.3.3


## 2. Konfigurasi

In [2]:
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'YOUR_OPENAI_KEY_HERE')

LLM_MODEL = 'gpt-4.1-mini'

TOP_K_RETRIEVAL = 5

DATASET_NAME   = 'qiaojin/PubMedQA'
DATASET_SUBSET = 'pqa_labeled'
MAX_SAMPLES    = 500

TEMPERATURE = 0.0
SEED        = 42

NOTEBOOK_DIR    = Path('.')
BM25_INDEX_PATH = NOTEBOOK_DIR / 'pubmedqa_bm25.pkl'
RESULTS_DIR     = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

CONFIG_NAME    = 'qr_openai'
PHASE1_PATH    = RESULTS_DIR / f'{CONFIG_NAME}_phase1_answers.json'

BASELINE_PHASE1_PATH = RESULTS_DIR / 'baseline_openai_phase1_answers.json'
BASELINE_PHASE2_PATH = RESULTS_DIR / 'baseline_openai_phase2_custom.json'

print('Konfigurasi:')
print(f'  LLM        : {LLM_MODEL} (via OpenAI API)')
print(f'  Retriever  : BM25 + Query Rewriting')
print(f'  Top-K      : {TOP_K_RETRIEVAL}')
print(f'  Sampel     : {MAX_SAMPLES}')
print(f'  Config     : {CONFIG_NAME}')

Konfigurasi:
  LLM        : gpt-4.1-mini (via OpenAI API)
  Retriever  : BM25 + Query Rewriting
  Top-K      : 5
  Sampel     : 500
  Config     : qr_openai


In [3]:
# ============================================================
# Setup OpenAI Client
# ============================================================

openai_client = OpenAI(api_key=OPENAI_API_KEY)

def openai_generate(prompt: str, max_tokens: int = 300, temperature: float = TEMPERATURE) -> str:
    for attempt in range(5):
        try:
            response = openai_client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
                seed=SEED,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                wait = (attempt + 1) * 10
                print(f'  [Rate limit] Tunggu {wait}s... (attempt {attempt+1}/5)')
                time.sleep(wait)
            elif '500' in err or '502' in err or '503' in err:
                wait = (attempt + 1) * 5
                print(f'  [Server error] Tunggu {wait}s... (attempt {attempt+1}/5)')
                time.sleep(wait)
            else:
                print(f'  [OpenAI Error] {type(e).__name__}: {err[:100]}')
                raise
    raise RuntimeError('OpenAI API gagal setelah 5 percobaan.')

print('Testing OpenAI API...')
_test = openai_generate('Reply with exactly: OK', max_tokens=5)
print(f'Response: {_test!r} | Model: {LLM_MODEL}')
print('OpenAI client siap!')

Testing OpenAI API...
Response: 'OK' | Model: gpt-4.1-mini
OpenAI client siap!


## 3. Data Classes dan Tokenizer BM25

In [4]:
@dataclass
class Document:
    text         : str
    pubid        : str
    question     : str
    section_label: str
    answer       : str
    decision     : str

@dataclass
class RetrievalResult:
    document: Document
    score   : float


def tokenize_bm25(text: str) -> List[str]:
    """Tokenizer untuk BM25: hapus tanda baca, lowercase, split spasi."""
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower()).split()


sample_text = 'Does aspirin (75mg) reduce myocardial infarction risk?'
print(f'Tokenisasi BM25: {tokenize_bm25(sample_text)}')
print('Data classes dan tokenizer siap.')

Tokenisasi BM25: ['does', 'aspirin', '75mg', 'reduce', 'myocardial', 'infarction', 'risk']
Data classes dan tokenizer siap.


## 4. Muat Dataset dan Bangun BM25 Index

BM25 index di-share dengan notebook baseline — gunakan file yang sama agar tidak rebuild dari scratch.

In [5]:
def load_pubmedqa(subset=DATASET_SUBSET, max_samples=MAX_SAMPLES):
    print(f'Memuat PubMedQA ({subset})...')
    dataset = load_dataset(DATASET_NAME, subset, trust_remote_code=True)
    data    = dataset['train']
    if max_samples and len(data) > max_samples:
        data = data.select(range(max_samples))
    print(f'Dimuat {len(data)} sampel')
    return data


def prepare_documents(data) -> List[Document]:
    docs = []
    for item in data:
        pubid = str(item['pubid'])
        for ctx, label in zip(item['context']['contexts'], item['context']['labels']):
            docs.append(Document(
                text=ctx.strip(), pubid=pubid,
                question=item['question'], section_label=label,
                answer=item['long_answer'], decision=item['final_decision']
            ))
    print(f'Total potongan dokumen: {len(docs)}')
    return docs


def load_or_build_bm25(data) -> Tuple[BM25Okapi, List[Document]]:
    """Muat BM25 index dari file jika ada (shared dengan baseline), atau bangun dari scratch."""
    if BM25_INDEX_PATH.exists():
        print(f'Memuat BM25 index dari {BM25_INDEX_PATH}...')
        with open(BM25_INDEX_PATH, 'rb') as f:
            saved = pickle.load(f)
        print(f'Dimuat: {len(saved["documents"])} dokumen')
        return saved['bm25'], saved['documents']
    else:
        print('Membangun BM25 index...')
        documents = prepare_documents(data)
        tokenized = [tokenize_bm25(d.text) for d in documents]
        bm25      = BM25Okapi(tokenized)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump({'bm25': bm25, 'documents': documents}, f)
        print(f'Index disimpan ke {BM25_INDEX_PATH}')
        return bm25, documents


t0 = time.time()
pubmedqa_data         = load_pubmedqa()
bm25_index, documents = load_or_build_bm25(pubmedqa_data)
print(f'Selesai dalam {time.time()-t0:.1f} detik')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Memuat PubMedQA (pqa_labeled)...


Dimuat 500 sampel
Memuat BM25 index dari pubmedqa_bm25.pkl...
Dimuat: 1706 dokumen
Selesai dalam 7.9 detik


## 5. Query Rewriting dengan LLM

LLM mereformulasi query asli menjadi pertanyaan yang lebih kaya terminologi medis untuk meningkatkan kualitas BM25 retrieval.

**Kenapa QR membantu BM25?**
- BM25 bergantung pada keyword matching — query yang lebih spesifik dan kaya terminologi meningkatkan recall dokumen relevan.
- LLM dapat mengekspansi singkatan, menambahkan sinonim medis, dan memperjelas konteks pertanyaan.

In [6]:
QUERY_REWRITE_PROMPT = (
    'You are a query rewriting assistant for a biomedical question-answering system.\n'
    'Rewrite the following medical question to improve retrieval from a PubMed research database.\n\n'
    'Rules:\n'
    '1. Be more specific and add relevant medical/scientific terminology.\n'
    '2. Expand abbreviations (e.g. "MI" -> "myocardial infarction").\n'
    '3. Preserve the original yes/no/maybe answerable intent.\n'
    '4. Output ONLY the rewritten question, no explanations.\n\n'
    'Original question: {query}\n\n'
    'Rewritten question:'
)

def rewrite_query(query: str) -> str:
    prompt = QUERY_REWRITE_PROMPT.format(query=query)
    try:
        rewritten = openai_generate(prompt, max_tokens=150, temperature=0.3)
        rewritten = rewritten.strip().replace('\n', ' ')
        return rewritten if len(rewritten) >= 10 else query
    except Exception as e:
        print(f'  [QR Error] {e} -- menggunakan query asli')
        return query

test_queries_qr = [
    'Does aspirin reduce the risk of myocardial infarction?',
    'Is vitamin D effective for COVID-19?',
    'Can exercise prevent T2DM?',
]
print('Contoh Query Rewriting:')
print('=' * 70)
for q in test_queries_qr:
    rw = rewrite_query(q)
    print(f'\nAsli    : {q}')
    print(f'Rewrite : {rw}')

Contoh Query Rewriting:

Asli    : Does aspirin reduce the risk of myocardial infarction?
Rewrite : Does aspirin administration reduce the risk of acute myocardial infarction in patients at risk for cardiovascular disease?

Asli    : Is vitamin D effective for COVID-19?
Rewrite : Is vitamin D supplementation effective in preventing or treating coronavirus disease 2019 (COVID-19) infection?

Asli    : Can exercise prevent T2DM?
Rewrite : Can regular physical exercise prevent the onset of type 2 diabetes mellitus?


## 6. Fungsi Retrieval dengan Query Rewriting (BM25)

In [7]:
def retrieve_with_qr(query: str, k: int = TOP_K_RETRIEVAL) -> Tuple[List[RetrievalResult], str]:
    """
    Retrieval menggunakan BM25 dengan Query Rewriting.
    Returns: (retrieved_docs, rewritten_query)
    """
    rewritten = rewrite_query(query)
    tokens    = tokenize_bm25(rewritten)
    scores    = bm25_index.get_scores(tokens)
    top_k     = np.argsort(scores)[::-1][:k]
    results   = [RetrievalResult(document=documents[i], score=float(scores[i])) for i in top_k]
    return results, rewritten


# Test
test_q = 'Does aspirin reduce the risk of myocardial infarction?'
test_r, test_rw = retrieve_with_qr(test_q)
print(f'Query asli : {test_q}')
print(f'Rewrite    : {test_rw}')
print(f'\nTop-{TOP_K_RETRIEVAL} dokumen (BM25 + QR):')
for i, r in enumerate(test_r, 1):
    print(f'  [{i}] Score={r.score:.4f} | {r.document.section_label} | {r.document.text[:90]}...')

Query asli : Does aspirin reduce the risk of myocardial infarction?
Rewrite    : Does aspirin administration reduce the risk of acute myocardial infarction in patients at risk for cardiovascular disease?

Top-5 dokumen (BM25 + QR):
  [1] Score=32.4434 | BACKGROUND | It has recently been shown that non-high density lipoprotein cholesterol (non-HDL-C) may b...
  [2] Score=31.6587 | METHODS | By use of the Cooperative Cardiovascular Project database (a retrospective medical record ...
  [3] Score=28.5935 | DESIGN | Within a prospective, population-based cohort study individuals without history of myocard...
  [4] Score=28.3344 | BACKGROUND | The role of early revascularization among patients with acute myocardial infarction compli...
  [5] Score=27.6832 | OBJECTIVE | Myocardial damage that is associated with percutaneous coronary intervention (PCI) partial...


## 7. Prompt Generasi dan Fungsi Generate

In [8]:
GENERATION_PROMPT = (
    'You are a medical research assistant. '
    'Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n'
    'Context from medical literature:\n{context}\n\n'
    'Question: {question}\n\n'
    'Instructions:\n'
    '- Carefully read the context and assess whether it supports or refutes the question.\n'
    '- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n'
    '- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n'
    '  - yes   : the evidence supports the hypothesis, even if not perfectly conclusive\n'
    '  - no    : the evidence refutes or does not support the hypothesis\n'
    '  - maybe : ONLY if the evidence is directly contradictory (some findings say yes,\n'
    '            others say no), or if the context contains no relevant information at all\n'
    '- IMPORTANT: If the evidence leans in one direction, even partially, choose yes or no.\n'
    '  Do NOT use maybe simply because the evidence is limited or not 100%% certain.\n\n'
    'Answer:'
)

def generate_answer(query: str, retrieved: List[RetrievalResult]) -> str:
    context = '\n\n'.join(
        f'[{i}] ({r.document.section_label}): {r.document.text}'
        for i, r in enumerate(retrieved, 1)
    )
    return openai_generate(
        GENERATION_PROMPT.format(context=context, question=query),
        max_tokens=300, temperature=TEMPERATURE
    )

test_ans = generate_answer(test_q, test_r)
print('Output generation:')
print('-' * 60)
print(test_ans)

Output generation:
------------------------------------------------------------
The provided abstracts focus on lipid predictors of cardiovascular risk, outcomes after acute myocardial infarction complicated by cardiogenic shock, revascularization strategies, and myocardial damage related to PCI. None of the abstracts mention aspirin or its effect on reducing the risk of myocardial infarction.

no


## 8. Ekstraksi Label yes/no/maybe

In [9]:
def extract_label(answer: str) -> str:
    """
    Ekstrak prediksi yes/no/maybe dari teks jawaban.
    Strategi (berurutan hingga ditemukan):
      1. Kata standalone di 3 baris terakhir (non-kosong)
      2. Kata standalone di seluruh teks
      3. Default ke 'maybe'
    """
    lines = [l.strip().lower() for l in answer.split('\n') if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r'[^a-z]', '', line)
        if word in ('yes', 'no', 'maybe'):
            return word
    for label in ('yes', 'no', 'maybe'):
        if re.search(r'\b' + label + r'\b', answer.lower()):
            return label
    return 'maybe'


cases = [
    ('Strong evidence.\nyes',    'yes'),
    ('No effect found.\nno',     'no'),
    ('Mixed results.\nmaybe',    'maybe'),
    ('Verdict: yes.',             'yes'),
    ('Totally unclear.',          'maybe'),
]
print('Unit test extract_label:')
all_ok = True
for txt, exp in cases:
    pred = extract_label(txt)
    ok   = pred == exp
    all_ok = all_ok and ok
    print(f'  [{"PASS" if ok else "FAIL"}] pred={pred!r} expected={exp!r}')
print(f'\nSemua lulus: {all_ok}')
print(f'Label dari test answer: {extract_label(test_ans)!r}')

Unit test extract_label:
  [PASS] pred='yes' expected='yes'
  [PASS] pred='no' expected='no'
  [PASS] pred='maybe' expected='maybe'
  [PASS] pred='yes' expected='yes'
  [PASS] pred='maybe' expected='maybe'

Semua lulus: True
Label dari test answer: 'no'


---
## Custom Evaluator 4 Metrik (Zero-NaN)

faithfulness, context_recall, answer_relevancy, context_precision — semua dijamin 0.0-1.0.

In [10]:
# ============================================================
# Custom Zero-NaN Evaluator -- 4 Metrik RAGAS (via OpenAI)
# Selalu return 0.0-1.0, TIDAK PERNAH NaN
# ============================================================

def _split_sentences(text: str) -> List[str]:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in parts if len(s.strip()) >= 15]

def _llm_yes_no(prompt: str) -> bool:
    try:
        resp = openai_generate(prompt, max_tokens=10, temperature=0.0)
        return 'yes' in resp.lower()[:15]
    except Exception:
        return False

def compute_faithfulness(answer: str, contexts: List[str]) -> float:
    sentences = _split_sentences(answer)
    if not sentences: return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = ('Context:\n{ctx}\n\nStatement: {sent}\n\n'
        'Is this statement directly supported by the context above? Answer with only "yes" or "no".')
    supported = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s)))
    return supported / len(sentences)

def compute_context_recall(reference: str, contexts: List[str]) -> float:
    sentences = _split_sentences(reference)
    if not sentences: return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = ('Context:\n{ctx}\n\nStatement: {sent}\n\n'
        'Is this statement supported by the context above? Answer with only "yes" or "no".')
    covered = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s)))
    return covered / len(sentences)

def compute_answer_relevancy(question: str, answer: str) -> float:
    sentences = _split_sentences(answer)
    if not sentences: return 0.0
    prompt_tmpl = ('Question: {question}\n\nStatement: {sent}\n\n'
        'Is this statement relevant to answering the question above? Answer with only "yes" or "no".')
    relevant = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(question=question, sent=s)))
    return relevant / len(sentences)

def compute_context_precision(question: str, contexts: List[str], reference: str) -> float:
    if not contexts: return 0.0
    prompt_tmpl = ('Question: {question}\n\nGround truth answer: {reference}\n\n'
        'Retrieved context: {ctx}\n\nDoes this context contain information useful for correctly '
        'answering the question based on the ground truth? Answer with only "yes" or "no".')
    relevance = []
    for ctx in contexts:
        is_rel = _llm_yes_no(prompt_tmpl.format(question=question, reference=reference[:300], ctx=ctx[:400]))
        relevance.append(1 if is_rel else 0)
    total_relevant = sum(relevance)
    if total_relevant == 0: return 0.0
    precision_sum, relevant_count = 0.0, 0
    for k, rel in enumerate(relevance):
        if rel:
            relevant_count += 1
            precision_sum += relevant_count / (k + 1)
    return precision_sum / total_relevant

def evaluate_custom(question, answer, contexts, reference):
    return {
        'faithfulness': compute_faithfulness(answer, contexts),
        'context_recall': compute_context_recall(reference, contexts),
        'answer_relevancy': compute_answer_relevancy(question, answer),
        'context_precision': compute_context_precision(question, contexts, reference),
    }

_ctx = ['Aspirin reduces blood clotting and is used for heart attack prevention.']
_ans = 'Aspirin helps prevent heart attacks. It works by reducing clotting.'
_ref = 'Aspirin is used for heart attack prevention by reducing blood clotting.'
_r = evaluate_custom('Does aspirin prevent heart attacks?', _ans, _ctx, _ref)
print(f'Smoke test (4 metrik): faith={_r["faithfulness"]:.2f} cr={_r["context_recall"]:.2f} ar={_r["answer_relevancy"]:.2f} cp={_r["context_precision"]:.2f}')
print('Zero-NaN evaluator siap (4 metrik via OpenAI).')

Smoke test (4 metrik): faith=1.00 cr=1.00 ar=1.00 cp=1.00
Zero-NaN evaluator siap (4 metrik via OpenAI).


---
## DEMO: Uji Coba 5 Sampel

Verifikasi pipeline QR berjalan dengan benar. Tampilkan query asli vs rewritten.

> Estimasi: ~3-10 menit (tergantung kecepatan Ollama)

In [10]:
DEMO_SIZE    = 5
demo_results = []

print(f'DEMO: {DEMO_SIZE} sampel pertama (RAG + Query Rewriting)')
print('=' * 70)

for i in range(DEMO_SIZE):
    s         = pubmedqa_data[i]
    q         = s['question']
    gt        = s['final_decision']
    ref       = s['long_answer']

    print(f'\n[{i+1}/{DEMO_SIZE}] PubID: {s["pubid"]}')
    print(f'  Q asli   : {q[:75]}...')
    print(f'  GT       : {gt}')

    t0 = time.time()
    retrieved, rewritten = retrieve_with_qr(q)
    t_ret = time.time() - t0
    print(f'  Q rewrite: {rewritten[:75]}...')

    t0     = time.time()
    answer = generate_answer(q, retrieved)
    t_gen  = time.time() - t0

    predicted = extract_label(answer)
    correct   = predicted == gt

    t0      = time.time()
    ctxs    = [r.document.text for r in retrieved]
    scores  = evaluate_ragas_single(q, answer, ctxs, ref)
    t_ragas = time.time() - t0

    demo_results.append({
        'idx': i, 'pubid': str(s['pubid']), 'question': q,
        'rewritten_query': rewritten,
        'ground_truth': gt, 'predicted_label': predicted,
        'is_correct': correct, 'answer': answer,
        'contexts': ctxs, 'reference': ref, **scores,
        't_retrieval': round(t_ret,2), 't_generation': round(t_gen,2), 't_ragas': round(t_ragas,2)
    })

    verdict = 'BENAR' if correct else 'SALAH'
    print(f'  Pred     : {predicted} [{verdict}]')
    print(f'  RAGAS    : faith={scores["faithfulness"]:.3f} | '
          f'rel={scores["answer_relevancy"]:.3f} | '
          f'cp={scores["context_precision"]:.3f} | '
          f'cr={scores["context_recall"]:.3f}')
    print(f'  Waktu    : ret={t_ret:.1f}s gen={t_gen:.1f}s ragas={t_ragas:.1f}s')

n_ok = sum(r['is_correct'] for r in demo_results)
print(f'\n{"="*70}')
print(f'RINGKASAN DEMO ({DEMO_SIZE} sampel -- RAG + QR):')
print(f'  Label Accuracy    : {n_ok}/{DEMO_SIZE} = {n_ok/DEMO_SIZE:.1%}')
print(f'  Hallucination Rate: {(DEMO_SIZE-n_ok)/DEMO_SIZE:.1%}')
for col in ['faithfulness','answer_relevancy','context_precision','context_recall']:
    print(f'  Avg {col:<22}: {np.nanmean([r[col] for r in demo_results]):.3f}')
print(f'{"="*70}')

---
## Fase 1: Generate Semua Jawaban (500 Sampel)

Retrieval + Query Rewriting + Generation. Rewritten query disimpan untuk analisis.
Disimpan inkremental setiap 10 sampel — aman untuk resume.

> Estimasi: ~35-70 menit untuk 500 sampel (sedikit lebih lambat dari baseline karena overhead QR per sampel)

In [11]:
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
        phase1_results = json.load(f)['results']
    start_from = len(phase1_results)
    print(f'Resume Fase 1: {start_from}/{MAX_SAMPLES} sudah selesai.')
else:
    phase1_results, start_from = [], 0
    print(f'Memulai Fase 1: {MAX_SAMPLES} sampel.')

if start_from < MAX_SAMPLES:
    print(f'Memproses {MAX_SAMPLES - start_from} sampel tersisa...\n')
    t_start = time.time()
    for i in range(start_from, MAX_SAMPLES):
        s          = pubmedqa_data[i]
        q, gt, ref = s['question'], s['final_decision'], s['long_answer']
        retrieved, rewritten = retrieve_with_qr(q)
        answer     = generate_answer(q, retrieved)
        predicted  = extract_label(answer)
        phase1_results.append({
            'idx': i, 'pubid': str(s['pubid']), 'question': q,
            'rewritten_query': rewritten,
            'ground_truth': gt, 'predicted_label': predicted,
            'is_correct': predicted == gt, 'answer': answer,
            'contexts': [r.document.text for r in retrieved],
            'reference': ref, 'retrieval_scores': [r.score for r in retrieved],
        })
        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, 'w', encoding='utf-8') as f:
                json.dump({'config': CONFIG_NAME,
                           'timestamp': datetime.now().isoformat(),
                           'max_samples': MAX_SAMPLES, 'completed': i+1,
                           'results': phase1_results}, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc  = sum(r['is_correct'] for r in phase1_results) / done
            eta  = (time.time()-t_start) / done * (MAX_SAMPLES-done) / 60
            print(f'  [{done:3d}/{MAX_SAMPLES}] Akurasi: {acc:.1%} | pred={predicted}, gt={gt} | ETA {eta:.1f} mnt')
    print(f'\nFase 1 selesai! Disimpan ke {PHASE1_PATH}')
else:
    print(f'Fase 1 sudah selesai ({MAX_SAMPLES} sampel).')

Memulai Fase 1: 500 sampel.
Memproses 500 sampel tersisa...

  [ 10/500] Akurasi: 50.0% | pred=yes, gt=yes | ETA 30.5 mnt
  [ 20/500] Akurasi: 65.0% | pred=yes, gt=yes | ETA 26.5 mnt
  [ 30/500] Akurasi: 66.7% | pred=yes, gt=yes | ETA 24.8 mnt
  [ 40/500] Akurasi: 60.0% | pred=yes, gt=no | ETA 23.6 mnt
  [ 50/500] Akurasi: 60.0% | pred=no, gt=no | ETA 22.8 mnt
  [ 60/500] Akurasi: 60.0% | pred=yes, gt=yes | ETA 21.7 mnt
  [ 70/500] Akurasi: 62.9% | pred=yes, gt=yes | ETA 21.3 mnt
  [ 80/500] Akurasi: 62.5% | pred=yes, gt=yes | ETA 21.1 mnt
  [ 90/500] Akurasi: 62.2% | pred=no, gt=maybe | ETA 20.5 mnt
  [100/500] Akurasi: 64.0% | pred=yes, gt=yes | ETA 20.2 mnt
  [110/500] Akurasi: 63.6% | pred=yes, gt=yes | ETA 19.6 mnt
  [120/500] Akurasi: 61.7% | pred=yes, gt=yes | ETA 19.0 mnt
  [130/500] Akurasi: 60.8% | pred=yes, gt=maybe | ETA 18.3 mnt
  [140/500] Akurasi: 60.7% | pred=yes, gt=yes | ETA 17.7 mnt
  [150/500] Akurasi: 60.7% | pred=yes, gt=no | ETA 17.4 mnt
  [160/500] Akurasi: 61.9

### Analisis Fase 1 (Label Accuracy)

In [12]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']
n         = len(results_p1)
n_correct = sum(r['is_correct'] for r in results_p1)
gts       = [r['ground_truth']    for r in results_p1]
preds     = [r['predicted_label'] for r in results_p1]

print(f'ANALISIS FASE 1 -- {n} sampel (RAG + Query Rewriting)')
print('=' * 55)
print(f'Label Accuracy    : {n_correct}/{n} = {n_correct/n:.1%}')
print(f'Hallucination Rate: {(n-n_correct)/n:.1%}\n')
print(f'  {"Label":<8} | {"Ground Truth":>12} | {"Prediksi":>12}')
print(f'  {"-"*8}-+{"-"*14}-+{"-"*12}')
for lbl in ['yes','no','maybe']:
    g, p = gts.count(lbl), preds.count(lbl)
    print(f'  {lbl:<8} | {g:>10} ({g/n:.0%}) | {p:>10} ({p/n:.0%})')
print('\nConfusion Matrix (baris=GT, kolom=Prediksi):')
lbls = ['yes','no','maybe']
print('  ' + f'{"GT/Pred":>8}' + ''.join(f'{l:>8}' for l in lbls))
for gt_l in lbls:
    row = f'  {gt_l:>8}'
    for pr_l in lbls:
        cnt = sum(1 for r in results_p1 if r['ground_truth']==gt_l and r['predicted_label']==pr_l)
        row += f'{cnt:>8}'
    print(row)

ANALISIS FASE 1 -- 500 sampel (RAG + Query Rewriting)
Label Accuracy    : 327/500 = 65.4%
Hallucination Rate: 34.6%

  Label    | Ground Truth |     Prediksi
  ---------+---------------+------------
  yes      |        275 (55%) |        312 (62%)
  no       |        159 (32%) |        146 (29%)
  maybe    |         66 (13%) |         42 (8%)

Confusion Matrix (baris=GT, kolom=Prediksi):
   GT/Pred     yes      no   maybe
       yes     222      36      17
        no      47      96      16
     maybe      43      14       9


---
## Phase 2: Evaluasi Custom 4 Metrik (500 Sampel)

Estimasi: ~30-60 menit. Resume otomatis.

In [13]:
MAX_CUSTOM_SAMPLES = 500
PHASE2_CUSTOM_PATH = RESULTS_DIR / f'{CONFIG_NAME}_phase2_custom.json'

with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    p1_custom = json.load(f)['results'][:MAX_CUSTOM_SAMPLES]

if PHASE2_CUSTOM_PATH.exists():
    with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
        p2_custom = json.load(f)['results']
    done_custom = {r['idx'] for r in p2_custom}
    print(f'Resume: {len(done_custom)}/{MAX_CUSTOM_SAMPLES} selesai.')
else:
    p2_custom, done_custom = [], set()
    print(f'Mulai: {MAX_CUSTOM_SAMPLES} sampel (custom zero-NaN, 4 metrik).')

remaining = [r for r in p1_custom if r['idx'] not in done_custom]
print(f'Sisa: {len(remaining)} sampel\n')

t0 = time.time()
for i, r in enumerate(remaining):
    scores = evaluate_custom(r['question'], r['answer'], r['contexts'], r['reference'])
    p2_custom.append({'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'], **scores})
    if (i+1) % 5 == 0 or i == len(remaining)-1:
        with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
            json.dump({'config': CONFIG_NAME, 'llm_model': LLM_MODEL,
                'timestamp': datetime.now().isoformat(), 'max_samples': MAX_CUSTOM_SAMPLES,
                'metrics': ['faithfulness','context_recall','answer_relevancy','context_precision'],
                'evaluator': 'custom_zero_nan_4metrics', 'results': p2_custom
            }, f, indent=2, ensure_ascii=False)
        done, total = i+1, len(remaining)
        eta = (time.time()-t0)/done*(total-done)/60 if done < total else 0
        avg_f  = sum(x['faithfulness'] for x in p2_custom)/len(p2_custom)
        avg_cr = sum(x['context_recall'] for x in p2_custom)/len(p2_custom)
        avg_ar = sum(x['answer_relevancy'] for x in p2_custom)/len(p2_custom)
        avg_cp = sum(x['context_precision'] for x in p2_custom)/len(p2_custom)
        print(f'  [{done:3d}/{total}] f={avg_f:.3f} cr={avg_cr:.3f} ar={avg_ar:.3f} cp={avg_cp:.3f} | ETA {eta:.1f}m')

n = len(p2_custom)
acc = sum(r['is_correct'] for r in p2_custom)/n
avg_f  = sum(r['faithfulness'] for r in p2_custom)/n
avg_cr = sum(r['context_recall'] for r in p2_custom)/n
avg_ar = sum(r['answer_relevancy'] for r in p2_custom)/n
avg_cp = sum(r['context_precision'] for r in p2_custom)/n
print(f'\nSelesai! {n} sampel:')
print(f'  Accuracy={acc:.1%%} | faith={avg_f:.4f} cr={avg_cr:.4f} ar={avg_ar:.4f} cp={avg_cp:.4f}')
print(f'\n  | {CONFIG_NAME} | {acc:.3f} | {1-acc:.3f} | {avg_f:.3f} | {avg_cr:.3f} | {avg_ar:.3f} | {avg_cp:.3f} |')

Mulai: 500 sampel (custom zero-NaN, 4 metrik).
Sisa: 500 sampel

  [  5/500] f=0.767 cr=0.850 ar=1.000 cp=0.707 | ETA 97.8m
  [ 10/500] f=0.817 cr=0.758 ar=1.000 cp=0.707 | ETA 95.1m
  [ 15/500] f=0.844 cr=0.806 ar=1.000 cp=0.629 | ETA 93.2m
  [ 20/500] f=0.883 cr=0.838 ar=1.000 cp=0.713 | ETA 92.7m
  [ 25/500] f=0.867 cr=0.870 ar=1.000 cp=0.739 | ETA 91.5m
  [ 30/500] f=0.878 cr=0.858 ar=1.000 cp=0.700 | ETA 87.7m
  [ 35/500] f=0.881 cr=0.879 ar=1.000 cp=0.724 | ETA 84.0m
  [ 40/500] f=0.896 cr=0.881 ar=0.992 cp=0.714 | ETA 83.1m
  [ 45/500] f=0.881 cr=0.861 ar=0.985 cp=0.683 | ETA 81.6m
  [ 50/500] f=0.893 cr=0.845 ar=0.980 cp=0.715 | ETA 80.2m
  [ 55/500] f=0.903 cr=0.814 ar=0.964 cp=0.684 | ETA 77.8m
  [ 60/500] f=0.911 cr=0.810 ar=0.967 cp=0.710 | ETA 76.7m
  [ 65/500] f=0.918 cr=0.787 ar=0.969 cp=0.705 | ETA 75.9m
  [ 70/500] f=0.924 cr=0.795 ar=0.957 cp=0.683 | ETA 75.4m
  [ 75/500] f=0.924 cr=0.798 ar=0.960 cp=0.684 | ETA 75.8m
  [ 80/500] f=0.929 cr=0.785 ar=0.963 cp=0.663 | E

ValueError: Invalid format specifier '.1%%' for object of type 'float'

---
## Analisis Kualitas Query Rewriting

Inspeksi kualitatif: cek contoh query asli vs rewritten dari Fase 1.

In [14]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    qr_p1 = json.load(f)['results']

print('Contoh Query Rewriting dari Fase 1:')
print('=' * 70)
for r in qr_p1[:10]:
    print(f'[{r["idx"]:3d}] Asli    : {r["question"][:75]}')
    print(f'       Rewrite : {r["rewritten_query"][:75]}')
    print(f'       GT={r["ground_truth"]} | Pred={r["predicted_label"]} | {"BENAR" if r["is_correct"] else "SALAH"}')
    print()

n_changed = sum(1 for r in qr_p1 if r['question'].strip().lower() != r['rewritten_query'].strip().lower())
print(f'Statistik QR:')
print(f'  Total sampel        : {len(qr_p1)}')
print(f'  Query berubah       : {n_changed} ({n_changed/len(qr_p1):.1%})')
print(f'  Query tidak berubah : {len(qr_p1)-n_changed} ({(len(qr_p1)-n_changed)/len(qr_p1):.1%})')

Contoh Query Rewriting dari Fase 1:
[  0] Asli    : Do mitochondria play a role in remodelling lace plant leaves during program
       Rewrite : Do mitochondria contribute to the regulation of programmed cell death and t
       GT=yes | Pred=yes | BENAR

[  1] Asli    : Landolt C and snellen e acuity: differences in strabismus amblyopia?
       Rewrite : Are there differences in visual acuity measurements between Landolt C and S
       GT=no | Pred=yes | SALAH

[  2] Asli    : Syncope during bathing in infants, a pediatric form of water-induced urtica
       Rewrite : Is syncope occurring during bathing in infants a pediatric manifestation of
       GT=yes | Pred=yes | BENAR

[  3] Asli    : Are the long-term results of the transanal pull-through equal to those of t
       Rewrite : Are the long-term clinical outcomes of transanal pull-through surgery compa
       GT=no | Pred=maybe | SALAH

[  4] Asli    : Can tailored interventions increase mammography use among HMO women?
       Rew